2. The Data Wrangling Workshop: Activity 6.01, page 309  

1. Read the visit_data.csv file.

In [5]:
import pandas as pd

url = "https://raw.githubusercontent.com/PacktWorkshops/The-Data-Wrangling-Workshop/refs/heads/master/Chapter06/datasets/visit_data.csv"
df = pd.read_csv(url, encoding = "utf-8")

2. Check for duplicates.

In [20]:
import pandas as pd
import mermaid
import os

def deduplicate():
    input_file_name = f"./" + input('Enter file name: ') + '.csv'
    df = pd.read_csv(input_file_name) # A (records)
    cols_input = input('Enter the column for which to deduplicate based on: ')
    cols = [c.strip() for c in cols_input.split(',')]
    folder = '_'.join(cols)
    os.makedirs(f"./{folder}/", exist_ok = True)
    
    output_file = f"./{folder}/{folder}_deduplicated"
    output_file_name = f"{output_file}.csv"
    output_for_recycle = f"./{folder}_deduplicated.csv"
    prisma_file_name = f"{output_file}.mmd"
    
    nulls_mask = df[cols].isnull().any(axis=1)
    df_nulls = df[nulls_mask] # B
    df_non_nulls = df[~nulls_mask] # C
    
    duplicates_mask = df_non_nulls.duplicated(subset = cols, keep = False)
    df_non_duplicates = df_non_nulls[~duplicates_mask] # D
    df_duplicates = df_non_nulls[duplicates_mask] # E
    #df_duplicates.groupby(cols, as_index=False).agg(agg_map)
    df_kept = df_duplicates.drop_duplicates(subset = cols, keep = 'first')
    #df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name in ['subgroup', 'source'] else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
    #df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name == 'subgroup' else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
    df_removed = df_duplicates[~df_duplicates.index.isin(df_kept.index)]
    #df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name == 'subgroup' and s.dropna().astype(str).str.strip().nunique() > 1 else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
    df_unique = df_non_nulls.drop_duplicates(subset = cols, keep = 'first') # df of unique
    df_deduplicated = pd.concat([df_non_duplicates, df_kept, df_nulls], ignore_index=True) # df of unique + df of non-duplicates
    
    results = {"records": len(df),  
    "nulls": len(df_nulls), 
    "non_nulls": len(df_non_nulls), 
    "non_duplicates": len(df_non_duplicates), 
    "duplicates": len(df_duplicates), 
    "removed": len(df_removed), 
    "kept": len(df_kept),
    "unique": len(df_unique),
    "deduplicated": len(df_deduplicated)
    }
    
    # output_file_name = deduplication/doi/doi_deduplicated.csv
    df_nulls.to_csv(output_file_name.replace('deduplicated','nulls'), index = False)
    df_deduplicated.to_csv(output_file_name, index = False)
    df_removed.to_csv(output_file_name.replace('deduplicated','duplicates_removed'), index = False)
    df_deduplicated.to_csv(output_for_recycle, index = False)
    
    graph_text = f"""
   
    graph TD;
    
    A["`**records** (*n* = {results['records']})`"];
    B["`null (*n* = {results['nulls']})`"];
    C["`non-null (*n* = {results['non_nulls']})`"];
    D["`non-duplicates (*n* = {results['non_duplicates']})`"];
    E["`duplicates (*n* = {results['duplicates']})`"];
    F["`duplicates kept (*n* = {results['kept']})`"];
    G["`duplicates removed (*n* = {results['removed']})`"];
    H["`unique (*n* = {results['unique']})`"];
    I["`deduplicated (*n* = {results['deduplicated']})`"];
    
    A --> B & C;
    C --> D & E;
    E --> F & G;
    D & F --> H
    B & H --> I"""
    
    with open(prisma_file_name, "w") as f:
        f.write(graph_text)
    
    !mmdc -i "{prisma_file_name}" -o "{output_file}"_light.svg
    !mmdc -i "{prisma_file_name}" -o "{output_file}"_dark.svg -t dark -b transparent
    print(results)

In [25]:
len(df['

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          1000 non-null   int64  
 1   first_name  704 non-null    object 
 2   last_name   704 non-null    object 
 3   email       1000 non-null   object 
 4   gender      495 non-null    object 
 5   ip_address  1000 non-null   object 
 6   visit       974 non-null    float64
dtypes: float64(1), int64(1), object(5)
memory usage: 54.8+ KB


In [24]:
deduplicate()

Enter file name:  visit_data
Enter the column for which to deduplicate based on:  ip_address


Generating single mermaid chart
Generating single mermaid chart
{'records': 1000, 'nulls': 0, 'non_nulls': 1000, 'non_duplicates': 1000, 'duplicates': 0, 'removed': 0, 'kept': 0, 'unique': 1000, 'deduplicated': 1000}


A deduplication script was used to evaluate the columns ip_address and email for duplicate rows and none were found for either. The functionality of the script was validated, albeit on a smaller sample of rows.

3. Check whether any essential column contains NaN.

I am not sure which column constitutes as essential, but there are null values (NaNs) for every column except id, email and ip_address.

4. Get rid of the outliers.
5. Report the size difference.
6. Create a box plot to check for outliers.
7. Get rid of any outliers.

3. Insert data into a SQL Lite database – create a table with the following data below that you will create yourself (Hint on how to create the SQL: Python for Data Analysis 2nd edition page 191, Python for Data Analysis 3rd Edition: Page 199):  
     a. Name, Address, City, State, Zip, Phone Number  
     b. Add at least 10 rows of data and submit your code with a query generating your results.  

```SQL

CREATE TABLE CONTACT (
    ID SERIAL PRIMARY KEY,
    Name varchar(30),
    Address varchar(255),
    City varchar(30),
    State varchar(2),
    Zip varchar(5),
    Phone_number varchar(10)
);

    
INSERT INTO CONTACT  ("ID","Name","Address","City","State","Zip","Phone_number")
VALUES (1,John Doe,30 Broadway,New York,NY,10000);
VALUES (2,Hinkle McCringleberry,123 Tomasza St,Afganistan,CA,90001);
VALUES (3,Danny Hufflepuff,10 Gaymor lane,NY,11779);
VALUES (4,Sylvia Griffindorf,59 Boulevard,MA,66666);
VALUES (5,Matthew Lionsbane,74 Statestreet,WA,19999);
VALUES (6,Fanny F,10 Gaymor lane,NY,11779);
VALUES (7,Janny V,10 Gaymor lane,NY,11779);
VALUES (8,Banny V,10 Gaymor lane,NY,11779);
VALUES (9,Ranny F,10 Gaymor lane,NY,11779);
VALUES (10,Manny E,10 Gaymor lane,NY,11779);
    

```